# 07 — Pruning vs. Scratch

## Reset vs revision under budget constraints

This notebook is designed to work in Colab even after a runtime restart.

It will:

1. locate or clone `pruning-rml`,
2. make `src/pruning_rml` importable,
3. create minimal helper files if they are missing,
4. save a comparison table,
5. save one figure.

Outputs:

```text
results/07_pruning_vs_scratch.csv
figures/07_pruning_vs_scratch.png
```

In [ ]:
# Cell 1 — robust repo setup

from pathlib import Path
import sys
import os
import subprocess

REPO_URL = "https://github.com/thinkthoughts/pruning-rml.git"
REPO_NAME = "pruning-rml"

cwd = Path.cwd().resolve()

if cwd.name == REPO_NAME:
    repo = cwd
elif cwd.name == "notebooks" and cwd.parent.name == REPO_NAME:
    repo = cwd.parent
elif (cwd / REPO_NAME).exists():
    repo = cwd / REPO_NAME
elif Path("/content/pruning-rml").exists():
    repo = Path("/content/pruning-rml")
else:
    target = Path("/content") if Path("/content").exists() else cwd
    os.chdir(target)
    subprocess.run(["git", "clone", REPO_URL], check=True)
    repo = target / REPO_NAME

os.chdir(repo)

src = repo / "src"
if str(src) not in sys.path:
    sys.path.insert(0, str(src))

print("repo:", repo)
print("src :", src)
print("src exists:", src.exists())

In [ ]:
# Cell 2 — guarantee minimal src helpers exist

pkg = repo / "src" / "pruning_rml"
pkg.mkdir(parents=True, exist_ok=True)

init_file = pkg / "__init__.py"
if not init_file.exists():
    init_file.write_text('__version__ = "0.1.0"\n', encoding="utf-8")

comparisons_file = pkg / "comparisons.py"
comparisons_file.write_text('''def label_mode(method):
    method = method.lower()
    if "scratch" in method or "random" in method:
        return "reset"
    if "prun" in method or "revision" in method:
        return "revision"
    return "observe"
''', encoding="utf-8")

paths_file = pkg / "paths.py"
paths_file.write_text('''from pathlib import Path

def ensure_dirs(root, names=("results", "figures", "reports")):
    root = Path(root)
    outputs = {}
    for name in names:
        path = root / name
        path.mkdir(parents=True, exist_ok=True)
        outputs[name] = path
    return outputs
''', encoding="utf-8")

for mod in ["pruning_rml", "pruning_rml.paths", "pruning_rml.comparisons"]:
    if mod in sys.modules:
        del sys.modules[mod]

print("src helpers ready")
print("files:", [p.name for p in pkg.iterdir()])

In [ ]:
# Cell 3 — imports

import pandas as pd
import matplotlib.pyplot as plt

from pruning_rml.comparisons import label_mode
from pruning_rml.paths import ensure_dirs

outputs = ensure_dirs(repo)
results = outputs["results"]
figures = outputs["figures"]

print("results:", results)
print("figures:", figures)

## Comparison schema

This is a qualitative orientation table, not a reproduction of paper benchmark values.

The goal is to make the core distinction explicit:

```text
scratch training
    =
reset

pruning + retraining
    =
revision
```

In [ ]:
# Cell 4 — build comparison table

rows = [
    {
        "method": "training from scratch",
        "mode": label_mode("training from scratch"),
        "starting_point": "random initialization",
        "training_budget": "matched tokens",
        "expected_behavior": "must rebuild capability",
        "rml_interpretation": "reset tests reconstruction",
        "revision_score": 1,
    },
    {
        "method": "pruned + retrained",
        "mode": label_mode("pruned + retrained"),
        "starting_point": "larger pretrained model",
        "training_budget": "matched tokens",
        "expected_behavior": "inherits useful structure",
        "rml_interpretation": "revision tests surviving residues",
        "revision_score": 3,
    },
    {
        "method": "scratch + full pipeline budget",
        "mode": label_mode("training from scratch"),
        "starting_point": "random initialization",
        "training_budget": "larger token budget",
        "expected_behavior": "can become competitive",
        "rml_interpretation": "reset can catch up with enough compute",
        "revision_score": 2,
    },
    {
        "method": "fine-grained pruning",
        "mode": label_mode("fine-grained pruning"),
        "starting_point": "larger pretrained model",
        "training_budget": "matched or continued budget",
        "expected_behavior": "may retain advantage longer",
        "rml_interpretation": "granular revision preserves more structure",
        "revision_score": 4,
    },
]

df = pd.DataFrame(rows)
df

In [ ]:
# Cell 5 — save CSV

csv_path = results / "07_pruning_vs_scratch.csv"
df.to_csv(csv_path, index=False)

print("saved:", csv_path)
print("exists:", csv_path.exists())

## Visual summary

The `revision_score` is a simple orientation score:

- lower = less inherited structure
- higher = more inherited structure / revision / residue preservation

Later notebooks can replace this with paper-derived measurements.

In [ ]:
# Cell 6 — save figure

fig_path = figures / "07_pruning_vs_scratch.png"

ax = df.plot.bar(
    x="method",
    y="revision_score",
    legend=False,
    figsize=(9, 5),
)

ax.set_title("Pruning vs. scratch: reset and revision")
ax.set_xlabel("Method")
ax.set_ylabel("Revision / inheritance score")
plt.xticks(rotation=20, ha="right")
plt.tight_layout()
plt.savefig(fig_path, dpi=160)
plt.show()

print("saved:", fig_path)
print("exists:", fig_path.exists())

In [ ]:
# Cell 7 — inspect outputs

for folder in ["results", "figures"]:
    path = repo / folder
    print(folder, "exists:", path.exists())
    if path.exists():
        print(" ", [p.name for p in path.iterdir()])

## Done

Notebook 07 worked if you see:

```text
results/07_pruning_vs_scratch.csv
figures/07_pruning_vs_scratch.png
```

Next notebook:

```text
notebooks/13_granularity.ipynb
```